# Pycistopic

In [19]:
import sys
sys.path

['/home/bt392/miniconda3/envs/scenicplus/lib/python3.8/site-packages/ray/thirdparty_files',
 '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/code/rna_atac/gene_regulatory_networks/scenicplus/epiblast_blood',
 '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/code/rna_atac/gene_regulatory_networks/scenicplus/epiblast_blood',
 '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/code/rna_atac/gene_regulatory_networks/scenicplus/epiblast_blood',
 '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/code/rna_atac/gene_regulatory_networks/scenicplus/epiblast_blood',
 '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/code/rna_atac/gene_regulatory_networks/scenicplus/epiblast_blood',
 '/home/bt392/miniconda3/envs/scenicplus/lib/python38.zip',
 '/home/bt392/miniconda3/envs/scenicplus/lib/python3.8',
 '/home/bt392/miniconda3/envs/scenicplus/lib/python3.8/lib-dynload',
 '',
 '/home/bt39

In [2]:
import warnings
warnings.simplefilter(action='ignore')
import pycisTopic
pycisTopic.__version__

'1.0.2.dev7+gda26088'

In [3]:
# Project directory
projDir = '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/results/rna_atac/gene_regulatory_networks/scenicplus/blood/'
# Output directory
outDir = projDir + 'pycistopic/'
import os
if not os.path.exists(outDir):
    os.makedirs(outDir)

In [4]:
work_dir = outDir

In [5]:
# Temp dir
tmpDir = '/home/bt392/ry/'

## 1. Creating a cisTopic object

In [4]:
# Create cisTopic object
from pycisTopic.cistopic_class import *
count_matrix=pd.read_csv(projDir+'epiblast_blood_atac_mtx.tsv', sep='\t')

In [5]:
count_matrix = count_matrix.set_index('cell')

In [6]:
#path_to_blacklist='/staging/leuven/stg_00002/lcb/cbravo/Multiomics_pipeline/pycisTopic/blacklist/hg19-blacklist.v2.bed'
cistopic_obj = create_cistopic_object(fragment_matrix=count_matrix) #, path_to_blacklist=path_to_blacklist)
# Adding cell information
cell_data =  pd.read_csv(projDir+'epiblast_blood_sample_metadata.tsv', sep='\t')
cell_data = cell_data.set_index('cell')
cistopic_obj.add_cell_data(cell_data)

2022-08-27 08:23:09,040 cisTopic     INFO     Converting fragment matrix to sparse matrix
2022-08-27 08:23:41,676 cisTopic     INFO     Creating CistopicObject
2022-08-27 08:23:45,136 cisTopic     INFO     Done!


In [7]:
print(cistopic_obj)

CistopicObject from project cisTopic with n_cells × n_regions = 12686 × 120828


In [27]:
# Save without doublets
with open(outDir + 'cisTopicObject.pkl', 'wb') as f:
  pickle.dump(cistopic_obj, f)

## 2. Run models

In [11]:
import os
path_to_mallet_binary='mallet'
# Run models
models= run_cgs_models( #run_cgs_models_mallet(path_to_mallet_binary, #run_cgs_models(
                    cistopic_obj,
                    n_topics=[2,3], #[2,4,10,15,25,35]
                    n_cpu=30,
                    n_iter=500,
                    random_state=555,
                    alpha=50,
                    alpha_by_topic=True,
                    eta=0.1,
                    eta_by_topic=False,
                    _temp_dir  =tmpDir, #Use SCRATCH if many models or big data set
                    save_path=None)
# Took about an hour? but didn't say when it was finished

2022-08-27 08:29:19,612	INFO services.py:1470 -- View the Ray dashboard at http://127.0.0.1:8265


(run_cgs_model pid=123993) 2022-08-27 08:29:25,232 cisTopic     INFO     Running model with 2 topics
(run_cgs_model pid=123987) 2022-08-27 08:29:25,575 cisTopic     INFO     Running model with 3 topics


In [16]:
path_to_mallet_binary="/rds/project/rds-SDzz0CATGms/users/bt392/miniconda3/envs/scenicplus/bin/mallet"
#"/rds/project/rds-SDzz0CATGms/users/bt392/miniconda3/envs/scenicplus/pkgs/mallet-2.0.8-ha770c72_0/bin/mallet" ?
# Run models
models= run_cgs_models_mallet(path_to_mallet_binary, #run_cgs_models(
                    cistopic_obj,
                    n_topics=[2,3], #[2,4,10,15,25,35]
                    n_cpu=30,
                    n_iter=500,
                    random_state=555,
                    alpha=50,
                    alpha_by_topic=True,
                    eta=0.1,
                    eta_by_topic=False,
                    tmp_path  =tmpDir, #Use SCRATCH if many models or big data set
                    save_path=None)

2022-08-27 09:43:56,381 cisTopic     INFO     Formatting input to corpus
2022-08-27 09:44:03,132 gensim.corpora.dictionary INFO     adding document #0 to Dictionary<0 unique tokens: []>
2022-08-27 09:45:16,613 gensim.corpora.dictionary INFO     adding document #10000 to Dictionary<0 unique tokens: []>
2022-08-27 09:45:41,692 gensim.corpora.dictionary INFO     built Dictionary<120828 unique tokens: ['0', '1', '2', '3', '4']...> from 12686 documents (total 167816377 corpus positions)
2022-08-27 09:45:41,695 cisTopic     INFO     Running model with 2 topics
2022-08-27 09:45:41,702 LDAMalletWrapper INFO     Serializing temporary corpus to /home/bt392/ry/corpus.txt
2022-08-27 09:50:49,323 LDAMalletWrapper INFO     Converting temporary corpus to MALLET format with /rds/project/rds-SDzz0CATGms/users/bt392/miniconda3/envs/scenicplus/bin/mallet import-file --preserve-case --keep-sequence --remove-stopwords --token-regex "\S+" --input /home/bt392/ry/corpus.txt --output /home/bt392/ry/corpus.mall

RuntimeError: command '['/rds/project/rds-SDzz0CATGms/users/bt392/miniconda3/envs/scenicplus/bin/mallet', 'train-topics', '--input', '/home/bt392/ry/corpus.mallet', '--num-topics', '2', '--alpha', '50', '--beta', '0.1', '--optimize-interval', '0', '--num-threads', '30', '--output-state', '/home/bt392/ry/4137ae_state.mallet.gz', '--output-doc-topics', '/home/bt392/ry/4137ae_doctopics.txt', '--output-topic-keys', '/home/bt392/ry/4137ae_topickeys.txt', '--num-iterations', '500', '--inferencer-filename', '/home/bt392/ry/4137ae_inferencer.mallet', '--doc-topics-threshold', '0.0', '--random-seed', '555']' return with error (code 1): b'[0.002s][warning][os,container] Duplicate cpuset controllers detected. Picking /sys/fs/cgroup/cpuset, skipping /cgroup-sl/cpuset.\nMallet LDA: 2 topics, 1 topic bits, 1 topic mask\nData loaded.\nException in thread "main" java.lang.OutOfMemoryError: Java heap space\n\tat cc.mallet.types.FeatureSequence.<init>(FeatureSequence.java:57)\n\tat cc.mallet.types.Featu

In [23]:
tmpDir = '/home/bt392/ry/b/'

In [ ]:
#path_to_mallet_binary="/rds/project/rds-SDzz0CATGms/users/bt392/miniconda3/envs/scenicplus/bin/mallet"
path_to_mallet_binary="/rds/project/rds-SDzz0CATGms/users/bt392/software/github/Mallet/bin/mallet"
os.environ['MALLET_MEMORY'] = '100G'
# Run models
models= run_cgs_models_mallet(path_to_mallet_binary, #run_cgs_models(
                    cistopic_obj,
                    n_topics=[2,3], #[2,4,10,15,25,35]
                    n_cpu=30,
                    n_iter=500,
                    random_state=555,
                    alpha=50,
                    alpha_by_topic=True,
                    eta=0.1,
                    eta_by_topic=False,
                    tmp_path  =tmpDir, #Use SCRATCH if many models or big data set
                    save_path=None)

2022-08-27 11:25:07,428 cisTopic     INFO     Formatting input to corpus
2022-08-27 11:25:14,056 gensim.corpora.dictionary INFO     adding document #0 to Dictionary<0 unique tokens: []>
2022-08-27 11:26:29,665 gensim.corpora.dictionary INFO     adding document #10000 to Dictionary<0 unique tokens: []>
2022-08-27 11:26:55,497 gensim.corpora.dictionary INFO     built Dictionary<120828 unique tokens: ['0', '1', '2', '3', '4']...> from 12686 documents (total 167816377 corpus positions)
2022-08-27 11:26:55,499 cisTopic     INFO     Running model with 2 topics
2022-08-27 11:26:55,504 LDAMalletWrapper INFO     Serializing temporary corpus to /home/bt392/ry/b/corpus.txt
2022-08-27 11:32:04,547 LDAMalletWrapper INFO     Converting temporary corpus to MALLET format with /rds/project/rds-SDzz0CATGms/users/bt392/software/github/Mallet/bin/mallet import-file --preserve-case --keep-sequence --remove-stopwords --token-regex "\S+" --input /home/bt392/ry/b/corpus.txt --output /home/bt392/ry/b/corpus.ma

In [ ]:
print(models)

[<pycisTopic.lda_models.CistopicLDAModel object at 0x2ad1b319ca60>, <pycisTopic.lda_models.CistopicLDAModel object at 0x2ad198ff6a00>]


In [30]:
%%bash
sbatch
#!/bin/bash
#SBATCH -p skylake-himem #skylake-himem #icelake #skylake #-himem #cclake
#SBATCH -A gottgens-sl2-cpu
### Modify this according to your Ray workload.
#SBATCH --nodes=1
#SBATCH --exclusive
#SBATCH --tasks-per-node=1
#SBATCH --time 12:00:00
### Modify this according to your Ray workload.
#SBATCH --cpus-per-task=30
#SBATCH --mem-per-cpu=5GB
#SBATCH --gpus-per-task=0

python /rds/project/rds-SDzz0CATGms/users/bt392/software/github/pycisTopic/model_scripts/runModels_lda_mallet.py \
        -i /rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/results/rna_atac/gene_regulatory_networks/scenicplus/blood/pycistopic/cisTopicObject.pkl \
        -o /rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/results/rna_atac/gene_regulatory_networks/scenicplus/blood/pycistopic/models_500_mallet.pkl \
        -nt 2,4,10,15,25,35 \
        -c 30 \
        -it 500 \
        -a 50 \
        -abt True \
        -e 0.1 \
        -ebt False \
        -sp /home/bt392/ry/b/intermediate_models \
        -s 555 \
        -td /home/bt392/ry/b

Submitted batch job 803448


In [ ]:
# Save
import pickle
with open(outDir+'models_500.pkl', 'wb') as f:
  pickle.dump(models, f)

## 3. Model selection

In [ ]:
#os.mkdir(outDir+'models')
model=evaluate_models(models,
                     select_model=None,
                     return_model=True,
                     metrics=['Arun_2010','Cao_Juan_2009', 'Minmo_2011', 'loglikelihood'],
                     plot_metrics=False,
                     save= outDir + 'models/model_selection.pdf')

In [ ]:
from pycisTopic.lda_models import *
model = evaluate_models(models,
                       select_model=16,
                       return_model=True,
                       metrics=['Arun_2010','Cao_Juan_2009', 'Minmo_2011', 'loglikelihood'],
                       plot_metrics=False)

In [ ]:
cistopic_obj.add_LDA_model(model)
pickle.dump(cistopic_obj,
            open(os.path.join(work_dir, 'scATAC/cistopic_obj.pkl'), 'wb'))

In [ ]:
# # Save
# with open(outDir + 'epiblast_blood_cisTopicObject.pkl', 'wb') as f:
#   pickle.dump(cistopic_obj, f)

## 4. Clustering & Visualisation

In [ ]:
# Load cisTopic object
import pickle
infile = open(outDir + 'epiblast_blood_cisTopicObject.pkl', 'rb')
cistopic_obj = pickle.load(infile)
infile.close()

In [ ]:
from pycisTopic.clust_vis import *
find_clusters(cistopic_obj,
                 target  = 'cell',
                 k = 10,
                 res = [0.6],
                 prefix = 'pycisTopic_',
                 scale = True,
                 split_pattern = '-')

In [ ]:
run_umap(cistopic_obj,
                 target  = 'cell', scale=True)

In [ ]:
run_tsne(cistopic_obj,
                 target  = 'cell', scale=True)

In [ ]:
os.mkdir(outDir+'/visualization')
plot_metadata(cistopic_obj,
                 reduction_name='UMAP',
                 variables=['celltype', 'stage', 'pycisTopic_leiden_10_0.6'], # Labels from RNA and new clusters
                 target='cell', num_columns=3,
                 text_size=10,
                 dot_size=5,
                 figsize=(15,5),
                 save= outDir + 'visualization/dimensionality_reduction_label.pdf')

In [ ]:
annot_dict={}
annot_dict['pycisTopic_leiden_10_0.6'] = {'0':'MM029 (0)', '1':'MM001 (1)', '2': 'MM034 (2)', '3': 'MM047 (3)', '4': 'MM011 (4)'}
cistopic_obj.cell_data['pycisTopic_leiden_10_0.6'] = [annot_dict['pycisTopic_leiden_10_0.6'][x] for x in cistopic_obj.cell_data['pycisTopic_leiden_10_0.6'].tolist()]

In [ ]:
plot_metadata(cistopic_obj,
                 reduction_name='UMAP',
                 variables=['celltype', 'stage', 'pycisTopic_leiden_10_0.6'], # Labels from RNA and new clusters
                 target='cell', num_columns=3,
                 text_size=10,
                 dot_size=5,
                 figsize=(15,5),
                 save= outDir + 'visualization/dimensionality_reduction_label.pdf')

In [ ]:
plot_topic(cistopic_obj,
            reduction_name = 'UMAP',
            target = 'cell',
            num_columns=5,
            save= outDir + 'visualization/dimensionality_reduction_topic_contr.pdf')

In [ ]:
cell_topic_heatmap(cistopic_obj,
                     variables = ['cellLine', 'LineType'],
                     scale = False,
                     legend_loc_x = 1.05,
                     legend_loc_y = -1.2,
                     legend_dist_y = -1,
                     figsize=(10,10),
                     save = outDir + 'visualization/heatmap_topic_contr.pdf')

In [ ]:
# Save
with open(outDir + 'epiblast_blood_cisTopicObject.pkl', 'wb') as f:
  pickle.dump(cistopic_obj, f)

## 5. Topic binarization & qc

In [ ]:
# Load cisTopic object
import pickle
infile = open(outDir + 'epiblast_blood_cisTopicObject.pkl', 'rb')
cistopic_obj = pickle.load(infile)
infile.close()

In [ ]:
os.mkdir(outDir+'topic_binarization')
from pycisTopic.topic_binarization import *
region_bin_topics = binarize_topics(cistopic_obj, method='otsu', ntop=3000, plot=True, num_columns=5, save= outDir + 'topic_binarization/otsu.pdf')

In [ ]:
binarized_cell_topic = binarize_topics(cistopic_obj, target='cell', method='li', plot=True, num_columns=5, nbins=60)

In [ ]:
from pycisTopic.topic_qc import *
topic_qc_metrics = compute_topic_metrics(cistopic_obj)

In [ ]:
fig_dict={}
fig_dict['CoherenceVSAssignments']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Log10_Assignments', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['AssignmentsVSCells_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Log10_Assignments', var_y='Cells_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSCells_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Cells_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSRegions_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Regions_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSMarginal_dist']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Marginal_topic_dist', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSGini_index']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Gini_index', var_color='Gini_index', plot=False, return_fig=True)

In [ ]:
# Plot topic stats in one figure
fig=plt.figure(figsize=(40, 43))
i = 1
for fig_ in fig_dict.keys():
    plt.subplot(2, 3, i)
    img = fig2img(fig_dict[fig_]) #To convert figures to png to plot together, see .utils.py. This converts the figure to png.
    plt.imshow(img)
    plt.axis('off')
    i += 1
plt.subplots_adjust(wspace=0, hspace=-0.70)
fig.savefig(outDir + 'topic_binarization/Topic_qc.pdf', bbox_inches='tight')
plt.show()

In [ ]:
topic_annot = topic_annotation(cistopic_obj, annot_var='celltype', binarized_cell_topic=binarized_cell_topic, general_topic_thr = 0.2)

In [ ]:
topic_annot

In [ ]:
topic_qc_metrics = pd.concat([topic_annot[['celltype', 'Ratio_cells_in_topic', 'Ratio_group_in_population']], topic_qc_metrics], axis=1)

In [ ]:
topic_qc_metrics

In [ ]:
# Save
with open(outDir + 'topic_binarization/Topic_qc_metrics_annot.pkl', 'wb') as f:
  pickle.dump(topic_qc_metrics, f)
with open(outDir + 'topic_binarization/binarized_cell_topic.pkl', 'wb') as f:
  pickle.dump(binarized_cell_topic, f)
with open(outDir + 'topic_binarization/binarized_topic_region.pkl', 'wb') as f:
  pickle.dump(region_bin_topics, f)

## 6. Differentially Accessible Regions (DARs)

In [ ]:
# Load cisTopic object
import pickle
infile = open(outDir + 'epiblast_blood_cisTopicObject.pkl', 'rb')
cistopic_obj = pickle.load(infile)
infile.close()

In [ ]:
from pycisTopic.diff_features import *
imputed_acc_obj = impute_accessibility(cistopic_obj, selected_cells=None, selected_regions=None, scale_factor=10**6)

In [ ]:
normalized_imputed_acc_obj = normalize_scores(imputed_acc_obj, scale_factor=10**4)

In [ ]:
os.mkdir(outDir+'DARs')
variable_regions = find_highly_variable_features(normalized_imputed_acc_obj,
                                           min_disp = 0.05,
                                           min_mean = 0.0125,
                                           max_mean = 3,
                                           max_disp = np.inf,
                                           n_bins=20,
                                           n_top_features=None,
                                           plot=True,
                                           save= outDir + 'DARs/HVR_plot.pdf')

In [ ]:
len(variable_regions)

In [ ]:
markers_dict= find_diff_features(cistopic_obj,
                      imputed_acc_obj,
                      variable='celltype',
                      var_features=variable_regions,
                      contrasts=None,
                      adjpval_thr=0.05,
                      log2fc_thr=np.log2(1.5),
                      n_cpu=5,
                      _temp_dir=tmpDir,
                      split_pattern = '-')

In [ ]:
from pycisTopic.clust_vis import *
plot_imputed_features(cistopic_obj,
                    reduction_name='UMAP',
                    imputed_data=imputed_acc_obj,
                    features=[markers_dict[x].index.tolist()[0] for x in ['MM034', 'MM011', 'MM029', 'MM047']],
                    scale=False,
                    num_columns=4,
                    save= outDir + 'DARs/example_best_DARs.pdf')

In [ ]:
x = [print(x + ': '+ str(len(markers_dict[x]))) for x in markers_dict.keys()]

In [ ]:
if not os.path.exists(os.path.join(work_dir, 'scATAC/candidate_enhancers')):
    os.makedirs(os.path.join(work_dir, 'scATAC/candidate_enhancers'))
import pickle
pickle.dump(markers_dict, open(os.path.join(work_dir, 'scATAC/candidate_enhancers/markers_dict.pkl'), 'wb'))
pickle.dump(imputed_acc_obj, open(outDir + 'DARs/Imputed_accessibility.pkl', 'wb'))

# Motif enrichment analysis using pycistarget

In [6]:
import pickle
region_bin_topics_otsu = pickle.load(open(os.path.join(work_dir, 'topic_binarization/binarized_topic_region.pkl'), 'rb'))
#region_bin_topics_top3k = pickle.load(open(os.path.join(work_dir, 'scATAC/candidate_enhancers/region_bin_topics_top3k.pkl'), 'rb'))
markers_dict = pickle.load(open(os.path.join(work_dir, 'candidate_enhancers/markers_dict.pkl'), 'rb'))

In [7]:
import pyranges as pr
from pycistarget.utils import region_names_to_coordinates
region_sets = {}
region_sets['topics_otsu'] = {}
# region_sets['topics_top_3'] = {}
region_sets['DARs'] = {}
for topic in region_bin_topics_otsu.keys():
    regions = region_bin_topics_otsu[topic].index[region_bin_topics_otsu[topic].index.str.startswith('chr')] #only keep regions on known chromosomes
    region_sets['topics_otsu'][topic] = pr.PyRanges(region_names_to_coordinates(regions))
# for topic in region_bin_topics_top3k.keys():
#     regions = region_bin_topics_top3k[topic].index[region_bin_topics_top3k[topic].index.str.startswith('chr')] #only keep regions on known chromosomes
#     region_sets['topics_top_3'][topic] = pr.PyRanges(region_names_to_coordinates(regions))
for DAR in markers_dict.keys():
    regions = markers_dict[DAR].index[markers_dict[DAR].index.str.startswith('chr')] #only keep regions on known chromosomes
    region_sets['DARs'][DAR] = pr.PyRanges(region_names_to_coordinates(regions))

In [8]:
for key in region_sets.keys():
    print(f'{key}: {region_sets[key].keys()}')

topics_otsu: dict_keys(['Topic1', 'Topic2', 'Topic3', 'Topic4', 'Topic5', 'Topic6', 'Topic7', 'Topic8', 'Topic9', 'Topic10', 'Topic11', 'Topic12', 'Topic13', 'Topic14', 'Topic15', 'Topic16', 'Topic17', 'Topic18', 'Topic19', 'Topic20', 'Topic21', 'Topic22', 'Topic23', 'Topic24', 'Topic25'])
DARs: dict_keys(['Allantois', 'Blood_progenitors_1', 'Blood_progenitors_2', 'Endothelium', 'Epiblast', 'Erythroid1', 'Erythroid2', 'Erythroid3', 'ExE_mesoderm', 'Haematoendothelial_progenitors', 'Mesenchyme', 'Mixed_mesoderm', 'Nascent_mesoderm', 'Primitive_Streak'])


In [9]:
db_fpath = "/rds/project/rds-SDzz0CATGms/users/bt392/software/cistarget_database/mm10"
motif_annot_fpath = "/rds/project/rds-SDzz0CATGms/users/bt392/software/cistarget_database/mm10"

In [10]:
rankings_db = os.path.join(db_fpath, 'mm10_screen_v10_clust.regions_vs_motifs.rankings.feather')
scores_db =  os.path.join(db_fpath, 'mm10_screen_v10_clust.regions_vs_motifs.scores.feather')
motif_annotation = os.path.join(motif_annot_fpath, 'motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl')

In [11]:
if not os.path.exists(os.path.join(work_dir, 'motifs')):
    os.makedirs(os.path.join(work_dir, 'motifs'))

In [12]:
from scenicplus.wrappers.run_pycistarget import run_pycistarget
run_pycistarget(
    region_sets = region_sets,
    species = 'mus_musculus',
    save_path = os.path.join(work_dir, 'motifs'),
    ctx_db_path = rankings_db,
    dem_db_path = scores_db,
    path_to_motif_annotations = motif_annotation,
    run_without_promoters = False,
    n_cpu = 30,
    _temp_dir = tmpDir,
    annotation_version = 'v10nr_clust',
    )

2022-08-28 19:23:25,025 pycisTarget_wrapper INFO     /rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/results/rna_atac/gene_regulatory_networks/scenicplus/blood/pycistopic/motifs folder already exists.
2022-08-28 19:23:25,442 pycisTarget_wrapper INFO     Loading cisTarget database for topics_otsu
2022-08-28 19:23:25,443 cisTarget    INFO     Reading cisTarget database
2022-08-28 19:25:22,359 pycisTarget_wrapper INFO     Running cisTarget for topics_otsu


2022-08-28 19:25:27,277	INFO services.py:1470 -- View the Ray dashboard at http://127.0.0.1:8265


(ctx_internal_ray pid=138386) 2022-08-28 19:25:33,188 cisTarget    INFO     Running cisTarget for Topic1 which has 9658 regions
(ctx_internal_ray pid=138410) 2022-08-28 19:25:33,823 cisTarget    INFO     Running cisTarget for Topic2 which has 4348 regions
(ctx_internal_ray pid=138408) 2022-08-28 19:25:34,653 cisTarget    INFO     Running cisTarget for Topic3 which has 7824 regions
(ctx_internal_ray pid=138411) 2022-08-28 19:25:35,006 cisTarget    INFO     Running cisTarget for Topic4 which has 7139 regions
(ctx_internal_ray pid=138395) 2022-08-28 19:25:35,761 cisTarget    INFO     Running cisTarget for Topic5 which has 13990 regions
(ctx_internal_ray pid=138409) 2022-08-28 19:25:36,346 cisTarget    INFO     Running cisTarget for Topic6 which has 6393 regions
(ctx_internal_ray pid=138397) 2022-08-28 19:25:37,058 cisTarget    INFO     Running cisTarget for Topic7 which has 9691 regions
(ctx_internal_ray pid=138402) 2022-08-28 19:25:37,965 cisTarget    INFO     Running cisTarget for Topic

2022-08-28 19:29:12,012	INFO services.py:1470 -- View the Ray dashboard at http://127.0.0.1:8265


(DEM_internal_ray pid=139772) 2022-08-28 19:29:19,199 DEM          INFO     Computing DEM for Topic2
(DEM_internal_ray pid=139773) 2022-08-28 19:29:19,584 DEM          INFO     Computing DEM for Topic1
(DEM_internal_ray pid=139761) 2022-08-28 19:29:20,530 DEM          INFO     Computing DEM for Topic3
(DEM_internal_ray pid=139767) 2022-08-28 19:29:21,036 DEM          INFO     Computing DEM for Topic4
(DEM_internal_ray pid=139766) 2022-08-28 19:29:22,130 DEM          INFO     Computing DEM for Topic6
(DEM_internal_ray pid=139760) 2022-08-28 19:29:23,263 DEM          INFO     Computing DEM for Topic5
(DEM_internal_ray pid=139757) 2022-08-28 19:29:23,214 DEM          INFO     Computing DEM for Topic8
(DEM_internal_ray pid=139762) 2022-08-28 19:29:23,662 DEM          INFO     Computing DEM for Topic7
(DEM_internal_ray pid=139768) 2022-08-28 19:29:24,421 DEM          INFO     Computing DEM for Topic9
(DEM_internal_ray pid=139755) 2022-08-28 19:29:25,159 DEM          INFO     Computing DEM f

2022-08-28 19:32:08,001	INFO services.py:1470 -- View the Ray dashboard at http://127.0.0.1:8265


(ctx_internal_ray pid=141013) 2022-08-28 19:32:13,822 cisTarget    INFO     Running cisTarget for Allantois which has 12574 regions
(ctx_internal_ray pid=141022) 2022-08-28 19:32:14,033 cisTarget    INFO     Running cisTarget for Blood_progenitors_1 which has 8294 regions
(ctx_internal_ray pid=141025) 2022-08-28 19:32:14,364 cisTarget    INFO     Running cisTarget for Blood_progenitors_2 which has 10821 regions
(ctx_internal_ray pid=141005) 2022-08-28 19:32:14,861 cisTarget    INFO     Running cisTarget for Endothelium which has 10816 regions
(ctx_internal_ray pid=141014) 2022-08-28 19:32:15,239 cisTarget    INFO     Running cisTarget for Epiblast which has 16791 regions
(ctx_internal_ray pid=141003) 2022-08-28 19:32:15,504 cisTarget    INFO     Running cisTarget for Erythroid1 which has 11628 regions
(ctx_internal_ray pid=141001) 2022-08-28 19:32:15,843 cisTarget    INFO     Running cisTarget for Erythroid2 which has 12250 regions
(ctx_internal_ray pid=141019) 2022-08-28 19:32:16,260 

2022-08-28 19:34:04,358	INFO services.py:1470 -- View the Ray dashboard at http://127.0.0.1:8265


(DEM_internal_ray pid=142229) 2022-08-28 19:34:11,597 DEM          INFO     Computing DEM for Blood_progenitors_1
(DEM_internal_ray pid=142230) 2022-08-28 19:34:12,043 DEM          INFO     Computing DEM for Allantois
(DEM_internal_ray pid=142228) 2022-08-28 19:34:12,318 DEM          INFO     Computing DEM for Blood_progenitors_2
(DEM_internal_ray pid=142226) 2022-08-28 19:34:12,670 DEM          INFO     Computing DEM for Endothelium
(DEM_internal_ray pid=142227) 2022-08-28 19:34:13,295 DEM          INFO     Computing DEM for Erythroid1
(DEM_internal_ray pid=142224) 2022-08-28 19:34:13,640 DEM          INFO     Computing DEM for Erythroid2
(DEM_internal_ray pid=142231) 2022-08-28 19:34:14,106 DEM          INFO     Computing DEM for Haematoendothelial_progenitors
(DEM_internal_ray pid=142232) 2022-08-28 19:34:14,291 DEM          INFO     Computing DEM for ExE_mesoderm
(DEM_internal_ray pid=142233) 2022-08-28 19:34:14,442 DEM          INFO     Computing DEM for Epiblast
(DEM_internal_ray

In [13]:
import dill
menr = dill.load(open(os.path.join(work_dir, 'motifs/menr.pkl'), 'rb'))